# 01 — Data Preparation
**Goal:** Load raw data, fix types, add useful features, save clean file.

Input : `data/petrol_pump_data_2024_2026.xlsx`  
Output: `data/clean_data.csv`

In [1]:
# Install once if needed
# !pip install pandas openpyxl matplotlib seaborn scikit-learn imbalanced-learn xgboost

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ')

Libraries loaded 


In [3]:
# ── Load data ──────────────────────────────────────────────────
FILE_PATH = '../data/petrol_pump_data_2024_2026.xlsx'

df = pd.read_excel(FILE_PATH)

print(f'Rows    : {df.shape[0]}')
print(f'Columns : {df.shape[1]}')
print(f'\nColumns: {df.columns.tolist()}')
df.head()

Rows    : 806
Columns : 14

Columns: ['Date', 'Day', 'Opening_Stock', 'MS_Sold', 'HSD1_Sold', 'HSD2_Sold', 'HSD3_Sold', 'Total_Sold', 'Closing_Stock', 'Cash', 'Online', 'Card', 'Dip', 'Refill_Required']


,Date,Day,Opening_Stock,MS_Sold,HSD1_Sold,HSD2_Sold,HSD3_Sold,Total_Sold,Closing_Stock,Cash,Online,Card,Dip,Refill_Required
0,01-01-2024,Monday,12000,564,1684,1219,899,4366,7634,155590,151727,90617,63,No
1,02-01-2024,Tuesday,7634,603,1740,1230,778,4351,3283,160056,131743,87654,27,No
2,03-01-2024,Wednesday,3283,441,1469,1174,867,3951,0,161531,127157,73959,1,Yes
3,04-01-2024,Thursday,12000,492,1632,1508,981,4613,7387,219210,127815,103870,61,No
4,05-01-2024,Friday,7387,485,1939,1368,1002,4794,2593,196707,153101,101182,21,No


In [4]:
# ── Fix data types & handle missing values ─────────────────────
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

missing = df.isnull().sum().sum()
print(f'Missing values: {missing}')
if missing > 0:
    df.fillna(method='ffill', inplace=True)
    print('Filled with forward-fill')
else:
    print('No missing values')

Missing values: 0
No missing values


In [5]:
# ── Add date features ──────────────────────────────────────────
df['Year']      = df['Date'].dt.year
df['Month']     = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek   # 0=Monday, 6=Sunday
df['Quarter']   = df['Date'].dt.quarter
df['Is_Weekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)

# Day name → number
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
df['Day_Num'] = df['Day'].map({d: i for i, d in enumerate(day_order)})

# Target: 1 = Refill needed, 0 = No refill
df['Target'] = (df['Refill_Required'] == 'Yes').astype(int)

print('Date features added ')
df[['Date','Year','Month','DayOfWeek','Is_Weekend','Target']].head()

Date features added 


,Date,Year,Month,DayOfWeek,Is_Weekend,Target
0,2024-01-01,2024,1,0,0,0
1,2024-01-02,2024,1,1,0,0
2,2024-01-03,2024,1,2,0,1
3,2024-01-04,2024,1,3,0,0
4,2024-01-05,2024,1,4,0,0


In [6]:
# ── Add business features ──────────────────────────────────────
# How much of the tank is left (0 = empty, 1 = full)
df['Stock_Ratio'] = (df['Closing_Stock'] / 12000).round(4)

# Average sales over last 7 days — captures weekly trend
df['Rolling_7d_Sales'] = df['Total_Sold'].rolling(window=7, min_periods=1).mean().round(0)

# Previous day's closing stock (how full were we yesterday?)
df['Prev_Closing'] = df['Closing_Stock'].shift(1).fillna(12000)

# Seasonal flags
df['Is_Festival_Month'] = df['Month'].isin([10, 11]).astype(int)  # Oct-Nov
df['Is_Monsoon_Month']  = df['Month'].isin([6, 7, 8]).astype(int) # Jun-Aug

print(f'Total features: {df.shape[1]}')
print(df.columns.tolist())

Total features: 26
['Date', 'Day', 'Opening_Stock', 'MS_Sold', 'HSD1_Sold', 'HSD2_Sold', 'HSD3_Sold', 'Total_Sold', 'Closing_Stock', 'Cash', 'Online', 'Card', 'Dip', 'Refill_Required', 'Year', 'Month', 'DayOfWeek', 'Quarter', 'Is_Weekend', 'Day_Num', 'Target', 'Stock_Ratio', 'Rolling_7d_Sales', 'Prev_Closing', 'Is_Festival_Month', 'Is_Monsoon_Month']


In [7]:
# ── Summary & Save ─────────────────────────────────────────────
print('=== Final Dataset Summary ===')
print(f'Rows       : {len(df)}')
print(f'Columns    : {df.shape[1]}')
print(f'Date range : {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'Refill days: {df["Target"].sum()} ({df["Target"].mean()*100:.1f}%)')
print(f'No refill  : {(df["Target"]==0).sum()} ({(df["Target"]==0).mean()*100:.1f}%)')

df.to_csv('../data/clean_data.csv', index=False)
print('\n Saved: data/clean_data.csv')

=== Final Dataset Summary ===
Rows       : 806
Columns    : 26
Date range : 2024-01-01 → 2026-03-16
Refill days: 303 (37.6%)
No refill  : 503 (62.4%)

 Saved: data/clean_data.csv
